# Cursor Frequency Demo

This notebook targets the `examples/cursor-frequency-demo` workspace. It generates Qblox sequence JSON files from the checked-in Q1ASM, configures two QCMs plus one QRM, runs a q1timeline preflight check, and starts the four sequencers used by the demo. The oscilloscope story runs as a triggered shot loop with acquisition feedback, a red cursor, and an orange sine retuned from the tracked center and measured magnitude.

In [ ]:
from __future__ import annotations

import json
import math
import os
import subprocess
import sys
import time
from pathlib import Path

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import yaml
from qcodes.instrument import find_or_create_instrument
from qblox_instruments import Cluster, ClusterType

ROOT = Path.cwd()
assert (ROOT / "q1timeline.yml").exists(), "Run this notebook from examples/cursor-frequency-demo"

def load_params() -> dict[str, int]:
    return json.loads((ROOT / "params.json").read_text(encoding="utf-8"))

def render_q1asm(name: str, params: dict[str, int]) -> str:
    return (ROOT / name).read_text(encoding="utf-8").format_map(params)

def gaussian(length: int, std_fraction: float = 0.18) -> list[float]:
    center = (length - 1) / 2
    std = max(length * std_fraction, 1.0)
    return [math.exp(-0.5 * ((index - center) / std) ** 2) for index in range(length)]

params = load_params()
params

## Connect to the cluster

Set `cluster_ip` to the hardware IP. Set it to `None` to use a dummy cluster with two QCMs and one QRM.

In [ ]:
cluster_ip = "10.10.200.53"
cluster_name = "QAS"

qcm_blue_module_index = 4
qcm_orange_module_index = 6
qrm_module_index = 8

dummy_cfg = (
    {
        2: ClusterType.CLUSTER_QCM,
        4: ClusterType.CLUSTER_QCM,
        6: ClusterType.CLUSTER_QRM,
    }
    if cluster_ip is None
    else None
)

cluster: Cluster = find_or_create_instrument(
    Cluster,
    recreate=True,
    name=cluster_name,
    identifier=cluster_ip,
    dummy_cfg=dummy_cfg,
)

cluster.reset()

qcm_modules = cluster.get_connected_modules(lambda mod: mod.is_qcm_type and not mod.is_rf_type)
qrm_modules = cluster.get_connected_modules(lambda mod: mod.is_qrm_type and not mod.is_rf_type)

qcm_blue_module = qcm_modules[qcm_blue_module_index]
qcm_orange_module = qcm_modules[qcm_orange_module_index]
qrm_module = qrm_modules[qrm_module_index]

def module_slot(module):
    slot = module.slot_idx
    return slot() if callable(slot) else slot

qcm_blue_slot = module_slot(qcm_blue_module)
qcm_orange_slot = module_slot(qcm_orange_module)
qrm_slot = module_slot(qrm_module)

print(cluster.get_system_status())
print("Blue QCM:", qcm_blue_module, "slot", qcm_blue_slot)
print("Orange QCM:", qcm_orange_module, "slot", qcm_orange_slot)
print("QRM:", qrm_module, "slot", qrm_slot)

## Generate sequence JSON

The first QCM sequencer 0 waits for the external trigger, emits QCM marker 1 for the scope trigger, and plays the blue peak on Ch0 with independent xorshift PRNG random walks for position and height. QRM sequencer 1 acquires left/right IQ samples, pops acquisition feedback IQ values, compares `abs(I)+abs(Q)`, updates `$MEAS_DELAY`, converts it to `$TRACKED_CENTER`, derives `$TRACKED_GAIN = (left magnitude + right magnitude) / 2`, and sends center and gain feedback to the red cursor and orange drive sequencers. QRM sequencer 0 centers the red cursor on the latest tracked center, applies a calibrated fixed-point cursor gain from the latest tracked gain, and receives the next center and gain for the next shot. The second QCM sequencer 0 receives the tracked center and gain, subtracts `TRACKER_MIN_CENTER`, scales the offset with `FREQ_CURSOR_SHIFT`, resets phase before each sine burst, retunes the low-MHz orange sine with `set_freq $FREQ_WORD`, and applies a calibrated fixed-point RF gain with `set_awg_gain`. Each Q1ASM program pads to `SHOT_PERIOD` and jumps back to `shot_loop:` for the triggered shot loop.

In [ ]:
PEAK_DUR = params["PEAK_DUR"]
ACQ_DUR = params["ACQ_DUR"]
CURSOR_DUR = params["CURSOR_DUR"]
SINE_DUR = params["SINE_DUR"]

cluster.clear_router()
cluster.set_cmm_route(params["CURSOR_CHANNEL"], [qrm_module.sequencer0])
cluster.set_cmm_route(params["FREQ_CHANNEL"], [qcm_orange_module.sequencer0])
cluster.set_cmm_route(params["CURSOR_GAIN_CHANNEL"], [qrm_module.sequencer0])
cluster.set_cmm_route(params["RF_GAIN_CHANNEL"], [qcm_orange_module.sequencer0])

blue_peak_pulse = gaussian(PEAK_DUR)
red_cursor_pulse = [1.0] * CURSOR_DUR
orange_envelope = [1.0] * SINE_DUR

sequences = {
    "blue_peak_sequence.json": {
        "waveforms": {"blue_peak": {"data": blue_peak_pulse, "index": 0}},
        "weights": {},
        "acquisitions": {},
        "program": render_q1asm("blue_peak.q1asm", params),
    },
    "red_cursor_sequence.json": {
        "waveforms": {"red_cursor": {"data": red_cursor_pulse, "index": 0}},
        "weights": {},
        "acquisitions": {},
        "program": render_q1asm("red_cursor.q1asm", params),
    },
    "red_tracker_sequence.json": {
        "waveforms": {},
        "weights": {},
        "acquisitions": {"tracked_edge": {"num_bins": params["EDGE_ACQ_NUM_BINS"], "index": params["EDGE_ACQ"]}},
        "program": render_q1asm("red_tracker.q1asm", params),
    },
    "orange_drive_sequence.json": {
        "waveforms": {"orange_envelope": {"data": orange_envelope, "index": 1}},
        "weights": {},
        "acquisitions": {},
        "program": render_q1asm("orange_drive.q1asm", params),
    },
}

for file_name, sequence in sequences.items():
    (ROOT / file_name).write_text(json.dumps(sequence, indent=4), encoding="utf-8")

print("Wrote", ", ".join(sequences))

## Optional timeline check

This validates the generated sequence files with the local Q1Lens/q1timeline analyzer and renderer before loading hardware.

In [ ]:
out_dir = ROOT / ".q1timeline"
out_dir.mkdir(exist_ok=True)

env = os.environ.copy()
repo_root = ROOT.parents[1]
env["PYTHONPATH"] = str(repo_root / "src")

sequence_json_by_id = {
    "blue_peak": "blue_peak_sequence.json",
    "red_cursor": "red_cursor_sequence.json",
    "red_tracker": "red_tracker_sequence.json",
    "orange_drive": "orange_drive_sequence.json",
}
preflight_project_data = yaml.safe_load((ROOT / "q1timeline.yml").read_text(encoding="utf-8"))
for sequencer in preflight_project_data["sequencers"]:
    sequencer["file"] = str((ROOT / sequencer["file"]).resolve())
    sequencer["sequence_json"] = str((ROOT / sequence_json_by_id[sequencer["id"]]).resolve())
preflight_project_data["params"]["file"] = str((ROOT / "params.json").resolve())
preflight_project = out_dir / "q1timeline.with-sequences.yml"
preflight_project.write_text(yaml.safe_dump(preflight_project_data, sort_keys=False), encoding="utf-8")

subprocess.run(
    [
        sys.executable,
        "-m",
        "q1lens",
        "q1timeline",
        "analyze",
        "--project",
        str(preflight_project),
        "--out",
        str(out_dir / "timeline_ir.json"),
        "--diagnostics",
        str(out_dir / "diagnostics.json"),
    ],
    cwd=ROOT,
    env=env,
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "q1lens",
        "q1timeline",
        "render",
        "--ir",
        str(out_dir / "timeline_ir.json"),
        "--out",
        str(out_dir / "timeline.html"),
        "--mode",
        "normal",
        "--no-open",
    ],
    cwd=ROOT,
    env=env,
    check=True,
)

ir = json.loads((out_dir / "timeline_ir.json").read_text(encoding="utf-8"))
diagnostics = json.loads((out_dir / "diagnostics.json").read_text(encoding="utf-8"))
assert not [item for item in diagnostics if item["severity"] == "error"]
assert not any(item["category"] == "feedback_latency_violation" for item in diagnostics)

expected_labels = {
    ("blue_peak", "play", "blue_peak"),
    ("red_cursor", "play", "red_cursor"),
    ("red_tracker", "acquire", "tracked_edge"),
    ("orange_drive", "play", "orange_envelope"),
}
event_labels = {
    (event["sequencer_id"], event["kind"], event["label"])
    for event in ir["events"]
}
assert expected_labels <= event_labels

flows = ir["feedback_flows"]
assert any(
    flow["source"] == "$TRACKED_CENTER" and flow["channel"] == str(params["CURSOR_CHANNEL"])
    for flow in flows
)
assert any(
    flow["source"] == "$TRACKED_CENTER" and flow["channel"] == str(params["FREQ_CHANNEL"])
    for flow in flows
)
assert any(
    flow["source"] == "$TRACKED_GAIN" and flow["channel"] == str(params["CURSOR_GAIN_CHANNEL"])
    for flow in flows
)
assert any(
    flow["source"] == "$TRACKED_GAIN" and flow["channel"] == str(params["RF_GAIN_CHANNEL"])
    for flow in flows
)

def first_event(sequencer_id: str, kind: str) -> dict:
    return next(
        event for event in ir["events"]
        if event["sequencer_id"] == sequencer_id and event["kind"] == kind
    )

blue_peak_play = first_event("blue_peak", "play")
red_cursor_play = first_event("red_cursor", "play")
orange_play = first_event("orange_drive", "play")
blue_peak_t0 = blue_peak_play["t0"]["value"]
blue_peak_t1 = blue_peak_t0 + blue_peak_play["duration"]["value"]
red_cursor_center = red_cursor_play["t0"]["value"] + red_cursor_play["duration"]["value"] / 2
orange_t0 = orange_play["t0"]["value"]
orange_t1 = orange_t0 + orange_play["duration"]["value"]
assert blue_peak_t0 < red_cursor_center < blue_peak_t1
assert orange_t0 < blue_peak_t1 and blue_peak_t0 < orange_t1

frequency_events = [
    event for event in ir["events"]
    if event["sequencer_id"] == "orange_drive"
    and event["kind"] == "latched_state_pending"
    and event["meta"].get("field") == "frequency"
]
assert len(frequency_events) == 1
assert "fb_pop_data channel" in frequency_events[0]["meta"]["value"]["expr"]
orange_update = next(
    event
    for event in ir["events"]
    if event["sequencer_id"] == "orange_drive"
    and event["kind"] == "upd_param"
    and event["meta"].get("applied_state", {}).get("frequency") == frequency_events[0]["meta"]["value"]
)
assert orange_update["meta"]["applied_state"]["frequency"] == frequency_events[0]["meta"]["value"]

print("Timeline events:", len(ir["events"]))
print("Feedback flows:", len(flows))
print("Validated acquisition feedback, tracked gain, and orange sine timeline.")
print("HTML:", out_dir / "timeline.html")

## Configure and start sequencers

The first QCM Ch0/1 carries the blue peak and QCM marker 1 carries the scope trigger. The second QCM Ch0/1 carries the orange drive. QRM IO0/1 carries the red cursor output and the tracker input used for acquisition feedback. The Cluster external trigger input is mapped to trigger address 1; without that trigger all four sequencers wait silently at `wait_trigger 1`. The default shot budget is 30 us, so keep the external trigger at or below 33.3 kHz while debugging.

The acquisition bins use rotating indices: each shot consumes two bins, so the first pass through `EDGE_ACQ_NUM_BINS = 65536` holds 32768 left/right shot pairs. At 33.3 kHz this is about 1.0 seconds before bin reuse. The default bench workflow starts once with `start_demo_burst()`, keeps the external trigger running, and downloads only in the final stop/reset cell via `finish_demo_burst(read_data=True)`. The final download does not wait for the full 65536-bin acquisition to complete, because normal bench runs usually stop with a partially filled allocation. Keep the run shorter than the first-pass window if bin-number order must remain a chronological shot history. Qblox binned acquisitions are accumulative: after wrap, a reused bin contains the hardware average of all writes to that bin and its `avg_cnt` increases; it is not overwritten with only the newest sample.

In [ ]:
def print_all_sequencer_statuses(label):
    print(label)
    for module_label, module in (
        ("QCM blue", qcm_blue_module),
        ("QCM orange", qcm_orange_module),
        ("QRM", qrm_module),
    ):
        for seq_idx in range(6):
            print(f"{module_label} slot {module_slot(module)} seq{seq_idx}:", module.get_sequencer_status(seq_idx))

def stop_demo_sequencers():
    cluster.stop_sequencer(slot=qcm_blue_slot, sequencer=0)
    cluster.stop_sequencer(slot=qcm_orange_slot, sequencer=0)
    cluster.stop_sequencer(slot=qrm_slot, sequencer=0)
    cluster.stop_sequencer(slot=qrm_slot, sequencer=1)

def clear_demo_runtime_state():
    qrm_module.delete_acquisition_data(1, all=True)
    for slot in (qcm_blue_slot, qcm_orange_slot, qrm_slot):
        for seq_idx in range(6):
            cluster.clear_sequencer_flags(slot=slot, sequencer=seq_idx)

def arm_demo_sequencers():
    cluster.arm_sequencer(slot=qcm_blue_slot, sequencer=0)
    cluster.arm_sequencer(slot=qcm_orange_slot, sequencer=0)
    cluster.arm_sequencer(slot=qrm_slot, sequencer=0)
    cluster.arm_sequencer(slot=qrm_slot, sequencer=1)

def start_demo_sequencers():
    cluster.start_sequencer(slot=qcm_blue_slot, sequencer=0)
    cluster.start_sequencer(slot=qcm_orange_slot, sequencer=0)
    cluster.start_sequencer(slot=qrm_slot, sequencer=0)
    cluster.start_sequencer(slot=qrm_slot, sequencer=1)

def pause_demo_triggers(idle_wait_s=0.01):
    cluster.ext_trigger_input_trigger_en(False)
    time.sleep(idle_wait_s)

def resume_demo_triggers():
    cluster.ext_trigger_input_trigger_address(1)
    cluster.ext_trigger_input_trigger_en(True)

def drain_tracker_acquisition(read_data=True, wait_for_status=False, clear_flags=False, as_numpy=True):
    if wait_for_status:
        qrm_module.get_acquisition_status(1, timeout=1)
    acquisitions = (
        cluster.get_acquisitions(qrm_slot, 1, as_numpy=as_numpy)
        if read_data
        else None
    )
    qrm_module.delete_acquisition_data(1, all=True)
    if clear_flags:
        cluster.clear_sequencer_flags(slot=qrm_slot, sequencer=1)
    return acquisitions

def read_cursor_register_snapshot(iteration=None):
    registers = cluster.get_sequencer_registers(qrm_slot, 0, ["R1", "R4", "R5"])
    snapshot = {
        "cursor_center": registers["R1"],
        "tracked_gain": registers["R4"],
        "cursor_output_gain": registers["R5"],
        "registers": registers,
    }
    if iteration is not None:
        snapshot["iteration"] = iteration
    return snapshot

def prepare_demo_run():
    cluster.ext_trigger_input_trigger_en(False)
    stop_demo_sequencers()
    clear_demo_runtime_state()
    arm_demo_sequencers()
    start_demo_sequencers()
    resume_demo_triggers()

def start_demo_burst():
    prepare_demo_run()
    print_all_sequencer_statuses("After start")

def finish_demo_burst(
    idle_wait_s=0.01,
    read_data=False,
    gate_external_trigger=True,
    wait_for_status=False,
    print_status=False,
    as_numpy=True,
):
    if gate_external_trigger:
        pause_demo_triggers(idle_wait_s)
    else:
        print("Stop the external trigger before running finish_demo_burst().")
        time.sleep(idle_wait_s)
    stop_demo_sequencers()
    acquisitions = drain_tracker_acquisition(
        read_data=read_data,
        wait_for_status=wait_for_status,
        clear_flags=True,
        as_numpy=as_numpy,
    )
    if print_status:
        print_all_sequencer_statuses("After burst finish")
    return acquisitions

def acquisition_magnitude(i_value, q_value):
    return math.hypot(i_value, q_value)

def tracker_acquisition_rows(acquisitions):
    tracked_edge = acquisitions["tracked_edge"]["acquisition"]["bins"]
    integration = tracked_edge["integration"]
    path0 = np.asarray(integration["path0"])
    path1 = np.asarray(integration["path1"])
    avg_cnt = np.asarray(tracked_edge.get("avg_cnt", np.zeros(len(path0), dtype=int)))

    left_bins = np.arange(0, max(len(path0) - 1, 0), 2)
    right_bins = left_bins + 1
    valid_pairs = (avg_cnt[left_bins] > 0) & (avg_cnt[right_bins] > 0)
    left_bins = left_bins[valid_pairs]
    right_bins = right_bins[valid_pairs]
    if len(left_bins) == 0:
        return []

    left_i = path0[left_bins]
    left_q = path1[left_bins]
    right_i = path0[right_bins]
    right_q = path1[right_bins]
    left_magnitude = np.hypot(left_i, left_q)
    right_magnitude = np.hypot(right_i, right_q)
    mean_magnitude = (left_magnitude + right_magnitude) / 2

    rows = []
    for index, left_bin in enumerate(left_bins.tolist()):
        right_bin = int(right_bins[index])
        mean_value = float(mean_magnitude[index])
        rows.append({
            "shot_index": int(left_bin // 2),
            "left_bin": int(left_bin),
            "right_bin": right_bin,
            "left_i": float(left_i[index]),
            "left_q": float(left_q[index]),
            "right_i": float(right_i[index]),
            "right_q": float(right_q[index]),
            "left_magnitude": float(left_magnitude[index]),
            "right_magnitude": float(right_magnitude[index]),
            "mean_magnitude": mean_value,
            "tracked_gain": mean_value,
            "left_avg_cnt": int(avg_cnt[left_bin]),
            "right_avg_cnt": int(avg_cnt[right_bin]),
        })
    return rows

def plot_tracker_acquisition(acquisitions, *, axes=None, title=None, cursor_history=None, max_plot_pairs=5000):
    rows = tracker_acquisition_rows(acquisitions)
    if not rows:
        raise ValueError("No complete tracked_edge acquisition pairs were returned.")
    plot_rows = rows
    if max_plot_pairs and len(rows) > max_plot_pairs:
        plot_step = math.ceil(len(rows) / max_plot_pairs)
        plot_rows = rows[::plot_step]
    x_values = [row["shot_index"] for row in plot_rows]
    show_cursor = bool(cursor_history)
    if axes is None:
        if show_cursor:
            fig, axes = plt.subplots(2, 1, figsize=(9, 6))
        else:
            fig, mag_ax = plt.subplots(1, 1, figsize=(9, 3.5))
            axes = np.asarray([mag_ax])
    else:
        axes = np.atleast_1d(axes)
        fig = axes[0].figure
        for axis in axes:
            axis.clear()

    mag_ax = axes[0]
    cursor_ax = axes[1] if show_cursor and len(axes) > 1 else None
    mag_ax.plot(x_values, [row["mean_magnitude"] for row in plot_rows], label="sqrt(I^2 + Q^2)")
    if len(plot_rows) != len(rows):
        mag_ax.text(
            0.01,
            0.95,
            f"showing {len(plot_rows)} of {len(rows)} pairs",
            transform=mag_ax.transAxes,
            va="top",
        )
    if cursor_ax is not None:
        cursor_x = [entry.get("iteration", index + 1) for index, entry in enumerate(cursor_history)]
        cursor_y = [entry["cursor_center"] for entry in cursor_history]
        cursor_ax.plot(cursor_x, cursor_y, marker="o", color="tab:purple", label="cursor center R1")
        cursor_ax.set_ylabel("cursor center")
        cursor_ax.set_xlabel("plot update")
        cursor_ax.legend()
    if title:
        mag_ax.set_title(title)
    mag_ax.set_ylabel("sqrt(I^2 + Q^2)")
    mag_ax.set_xlabel("tracked-edge shot pair")
    mag_ax.legend()
    fig.tight_layout()
    return fig, axes, rows

def run_demo_live_plot_loop(update_s=5.0, repeats=None, idle_wait_s=0.01):
    prepare_demo_run()
    print_all_sequencer_statuses("After live loop start")

    fig, axes = plt.subplots(2, 1, figsize=(9, 6))
    display_handle = display(fig, display_id=True)
    history = []
    cursor_history = []
    iteration = 0
    triggers_paused = False
    try:
        while repeats is None or iteration < repeats:
            iteration += 1
            time.sleep(update_s)
            pause_demo_triggers(idle_wait_s)
            triggers_paused = True
            acquisitions = drain_tracker_acquisition(read_data=True)
            cursor_snapshot = read_cursor_register_snapshot(iteration=iteration)
            cursor_history.append(cursor_snapshot)
            fig, axes, rows = plot_tracker_acquisition(
                acquisitions,
                axes=axes,
                title=f"Tracker acquisition update {iteration}",
                cursor_history=cursor_history,
            )
            display_handle.update(fig)
            history.append({
                "iteration": iteration,
                "rows": rows,
                "acquisitions": acquisitions,
                "cursor_snapshot": cursor_snapshot,
            })
            resume_demo_triggers()
            triggers_paused = False
    except KeyboardInterrupt:
        print("Live plot loop interrupted. Sequencers are still running; use the stop/reset cell when done.")
    finally:
        if triggers_paused:
            resume_demo_triggers()
    return history

def run_demo_burst_loop(burst_s=5.0, repeats=None, idle_wait_s=0.01):
    fig, axes = plt.subplots(2, 1, figsize=(9, 6))
    display_handle = display(fig, display_id=True)
    history = []
    cursor_history = []
    iteration = 0
    try:
        while repeats is None or iteration < repeats:
            iteration += 1
            start_demo_burst()
            time.sleep(burst_s)
            acquisitions = finish_demo_burst(idle_wait_s=idle_wait_s, read_data=True)
            cursor_snapshot = read_cursor_register_snapshot(iteration=iteration)
            cursor_history.append(cursor_snapshot)
            fig, axes, rows = plot_tracker_acquisition(
                acquisitions,
                axes=axes,
                title=f"Tracker acquisition burst {iteration}",
                cursor_history=cursor_history,
            )
            display_handle.update(fig)
            history.append({
                "iteration": iteration,
                "rows": rows,
                "acquisitions": acquisitions,
                "cursor_snapshot": cursor_snapshot,
            })
    except KeyboardInterrupt:
        finish_demo_burst(idle_wait_s=idle_wait_s, read_data=False)
    return history

qcm_blue_module.stop_sequencer()
qcm_orange_module.stop_sequencer()
qrm_module.stop_sequencer()

cluster.ext_trigger_input_trigger_en(False)
cluster.ext_trigger_input_trigger_address(1)

qcm_blue_module.disconnect_outputs()
qcm_orange_module.disconnect_outputs()
qrm_module.disconnect_outputs()
qrm_module.disconnect_inputs()

qcm_blue_module.sequencer0.connect_sequencer("out0_1")
qcm_orange_module.sequencer0.connect_sequencer("out0_1")
qrm_module.sequencer0.connect_sequencer("io0_1")
qrm_module.sequencer1.connect_sequencer("io0_1")

qrm_module.sequencer1.demod_en_acq(True)
qrm_module.sequencer1.integration_length_acq(ACQ_DUR)
qcm_blue_module.sequencer0.mod_en_awg(False)
qcm_orange_module.sequencer0.mod_en_awg(True)
qrm_module.sequencer0.mod_en_awg(False)

qcm_blue_module.sequencer0.sequence("blue_peak_sequence.json")
qrm_module.sequencer0.sequence("red_cursor_sequence.json")
qrm_module.sequencer1.sequence("red_tracker_sequence.json")
qcm_orange_module.sequencer0.sequence("orange_drive_sequence.json")

for seq in (qcm_blue_module.sequencer0, qcm_orange_module.sequencer0, qrm_module.sequencer0, qrm_module.sequencer1):
    seq.sync_en(True)

# Default bench workflow: start once, keep the external trigger running, then use the final stop/download cell.
# Optional monitoring: run_demo_live_plot_loop(...) downloads at every plot update and adds dead time.

start_demo_burst()


## Stop and reset

In [ ]:
print("Stopping sequencers and downloading tracker acquisition...")
download_t0 = time.perf_counter()
tracker_acquisitions = finish_demo_burst(read_data=True, print_status=False)
download_s = time.perf_counter() - download_t0
print(f"Download finished in {download_s:.3f} s.")

print("Plotting tracker acquisition...")
plot_t0 = time.perf_counter()
fig, axes, rows = plot_tracker_acquisition(tracker_acquisitions)
display(fig)
plt.close(fig)
plot_s = time.perf_counter() - plot_t0
print(f"Plotted {len(rows)} tracked-edge pairs in {plot_s:.3f} s.")

# Optional full hardware reset after inspecting the plot:
# cluster.reset()
# print(cluster.get_system_status())


In [ ]:
# Optional: run this only after the acquisition plot is visible.
print("Reading cursor register snapshot...")
cursor_snapshot = read_cursor_register_snapshot(iteration=1)
fig, axes, rows = plot_tracker_acquisition(
    tracker_acquisitions,
    cursor_history=[cursor_snapshot],
)
display(fig)
plt.close(fig)
print("Cursor snapshot:", cursor_snapshot)
